In [4]:
# [역할] 실습 환경 준비: 프로젝트 루트로 이동하고, 재료·모델 경로를 정하고, backend를 import 경로에 추가
import os        # 작업 폴더 변경용
import pathlib   # 경로를 객체로 다루기
import sys       # 모듈 검색 경로(sys.path) 조작용

here = pathlib.Path.cwd()                                                  # 현재 작업 폴더
# ROOT = next(p for p in (here, *here.parents) if (p / 'backend').is_dir())  # 위로 올라가며 backend 폴더가 있는 곳 = 프로젝트 루트
ROOT = here.parents[2] if here.name == "Fri" else here      # Fri에서 실행했으면 3단계 위가 프로젝트 루트
os.chdir(ROOT)                                              # 작업 폴더를 루트로 이동
SANDBOX = ROOT / "sandbox" / "W5" / "Fri"                   # 오늘 실습 폴더
SAMPLES = ROOT / "sandbox" / "W5" / "Mon" / "samples"       # 샘플 문서(재료) 폴더
MODELS = ROOT.parent / 'models'                             # 임베딩 모델 폴더 (프로젝트 옆)
FORMATS = SAMPLES / '_formats'                              # 여러 형식 샘플 폴더

print("프로젝트 루트  :", ROOT)       # 루트 경로 확인
print("재료 폴더      :", SAMPLES)    # 샘플 폴더 경로 확인

if str(ROOT / 'backend') not in sys.path:          # backend가 import 경로에 없으면
    sys.path.insert(0, str(ROOT / 'backend'))      # 맨 앞에 추가 → `from app...` import 가능

프로젝트 루트  : d:\hanwha-agent
재료 폴더      : d:\hanwha-agent\sandbox\W5\Mon\samples


In [5]:
# [역할] 오늘 시험할 파일 경로 정해 두기
PDF=SAMPLES / 'DOC-HR-014_국내출장_여비_규정_v2.0.pdf'        # 일반 PDF (글자 있음, 표 7개)
DOCX=SAMPLES / 'DOC-HR-014_국내출장_여비_규정_v2.0.docx'      # 같은 문서의 Word 원본 (비교 기준)
SCANNED=SAMPLES / '_formats' / 'sample_scanned.pdf'           # 스캔본 PDF (글자 데이터 없음)

In [ ]:
# [역할] 마크다운 파서 확인: md 파일이 제목(#) 단위 블록으로 나뉘고, 제목이 위치(locator)가 되는지 출력
import importlib                                   # 모듈 다시 불러오기용 (코드 수정 후 reload)
import sandbox.W5.Fri.old_local_parsers as lp                 # 로컬 파서 모듈 (뒤 셀에서 사용)
from app.rag.chunker import chunk, summarize       # 청크 나누기, 청크 수 요약
from app.rag.parser import parse                   # 형식별로 파서를 골라 주는 입구

MD=SAMPLES / '_formats' / 'sample_guide.md'        # 마크다운 샘플
doc=parse(MD)                                      # 파싱 → .md 는 _parse_markdown 으로
print(f'블록 {len(doc.blocks)} -> {summarize(chunk(doc))}')   # 블록 수 → 청크 요약

for b in doc.blocks:                               # 블록마다
    print(f'[{b.locator}] {b.text}')               # [제목] 본문

블록 3 -> 조항 단위 1 + 표 단위 0 = 1 청크
[1. 신청] 출장 3일 전까지 그룹웨어에서 신청합니다.
[2. 승인] 팀장 승인 후 확정됩니다. 5일 초과 출장은 본부장 승인이 필요합니다.
[3. 정산] 복귀 후 7일 이내에 영수증을 첨부해 정산합니다.


HTML 파일

In [ ]:
# [역할] HTML 파서 확인: <head> 안의 <title> 글자가 본문 블록에 섞이지 않는지 확인 (True면 아직 섞임)
HTML=FORMATS / 'sample_page.html'                      # HTML 샘플
raw=HTML.read_text(encoding="utf-8")                   # HTML 원문
title=raw.split('<title>')[1].split('</title>')[0]     # <title> 추출
body=parse(HTML).blocks[0].text                        # 파싱 → .html 은 _parse_html 로
print(title)                                           # 제목 글자
print(body[:20])                                       # 본문 블록 앞 20자
print(body.startswith(title))                          # 본문이 제목으로 시작하나 → False 가 정상

중간에 날아간 셀이 있을지도...

XLSX

In [ ]:
# [역할] 병합 셀 해결 확인: A2:A3 병합 엑셀을 만들어, 아래 칸(A3)도 '인프라사업부'로 채워지는지 확인
from openpyxl import Workbook                        # 새 엑셀 파일 만들기용

SANDBOX.mkdir(parents=True, exist_ok=True)           # 실습 폴더가 없으면 만들기
wb = Workbook()                                      # 빈 통합문서
ws = wb.active                                       # 기본 시트
ws.title = "출장비 집계"                               # 시트 이름
ws.append(["구분", "1분기", "2분기"])                   # 머리행
ws.append(["인프라사업부", 4200000, 3800000])           # 2행
ws.append([None, 1200000, 1400000])                  # 3행 (A3는 병합될 칸이라 비워 둠)
ws.append(["구매팀", 900000, 1000000])                  # 4행
ws.merge_cells("A2:A3")                              # A2~A3 병합 → 값은 A2에만 있음
MERGED = SANDBOX / "merged.xlsx"                     # 저장 경로
wb.save(MERGED)                                      # 저장

doc = lp.parse_xlsx(MERGED)                          # 파싱 (_fill_merged 가 병합 칸을 채움)
print(f"블록 {len(doc.blocks)} · 표 {doc.table_count}")   # 블록·표 수
print(doc.blocks[0].text)                            # 3행에도 '인프라사업부'가 나오면 성공

In [ ]:
# [역할] 수식 칸 확인: 계산값이 저장 안 된 수식은 data_only 로 None 이 되는데, 파서가 수식 원문으로 표시하는지 확인
from openpyxl import Workbook                        # 새 엑셀 파일 만들기용
from openpyxl import load_workbook                   # 엑셀 읽기용

wb = Workbook()                                      # 빈 통합문서
ws = wb.active                                       # 기본 시트
ws.title = "부서별"                                   # 시트 이름
for row in (["부서", "출장비"], ["인프라사업부", 4200000], ["구매팀", 1200000], ["합계", "=SUM(B2:B3)"]):   # 마지막 행 B5 = 수식
    ws.append(row)                                   # 한 행씩 추가
FORMULA = SANDBOX / "formula.xlsx"                   # 저장 경로
wb.save(FORMULA)                                     # 저장 (파이썬은 수식을 계산하지 않음 → 계산값 없음)

print("data_only 로 읽은 B5 :", load_workbook(FORMULA, data_only=True).active["B5"].value)   # → None
print(lp.parse_xlsx(FORMULA).blocks[0].text)         # _fill_formulas 가 '=SUM(...) (계산값 없음)' 으로 채움

In [ ]:
# [역할] 제목줄·2단 머리글 확인: 맨 위 병합 제목은 표 제목으로, 두 줄 머리글은 '1분기 출장비' 처럼 한 줄로 합쳐지는지 확인
wb = Workbook()                                      # 빈 통합문서
ws = wb.active                                       # 기본 시트
ws.title = "분기별"                                   # 시트 이름
for row in (["2026 출장 · 교육비 집계", None, None, None, None],   # 1행: 표 제목 (A1:E1 병합 예정)
            ["구분", "1분기", None, "2분기", None],                # 2행: 위 머리글 (분기)
            [None, "출장비", "교육비", "출장비", "교육비"],          # 3행: 아래 머리글 (항목)
            ["인프라사업부", 4200000, 1200000, 3800000, 1400000],  # 데이터
            ["구매팀", 900000, 300000, 1000000, 250000]):          # 데이터
    ws.append(row)                                   # 한 행씩 추가
for rng in ("A1:E1", "B2:C2", "D2:E2", "A2:A3"):     # 제목줄, 분기 머리글, '구분' 세로 병합
    ws.merge_cells(rng)                              # 병합
HEADER = SANDBOX / "header.xlsx"                     # 저장 경로
wb.save(HEADER)                                      # 저장

print(lp.parse_xlsx(HEADER).blocks[0].text)          # 제목 + 한 줄로 합쳐진 머리글 표

In [ ]:
# [역할] 한 시트에 표 2개 확인: 빈 행을 경계로 '요약 · 표1', '요약 · 표2' 두 블록으로 나뉘는지 확인
wb = Workbook()                                      # 빈 통합문서
ws = wb.active                                       # 기본 시트
ws.title = "요약"                                     # 시트 이름
for row in (["항목", "상한"], ["숙박", "100,000"], ["식비", "35,000"], [],   # 첫 번째 표 + 빈 행(경계)
            ["직급", "일비", "숙박비", "식비"], ["1~2급", "40,000", "100,000", "35,000"]):   # 두 번째 표
    ws.append(row)                                   # 한 행씩 추가
MULTI = SANDBOX / "multi.xlsx"                       # 저장 경로
wb.save(MULTI)                                       # 저장

doc = lp.parse_xlsx(MULTI)                           # 파싱 (_split_regions 가 표를 나눔)
print(f"블록 {len(doc.blocks)} · 쪽 {doc.page_count} · 표 {doc.table_count}")   # 블록 2 · 표 2 가 정상
for b in doc.blocks:                                 # 블록마다
    print(f"--- [{b.kind}] {b.locator}")             # 종류와 위치
    print(b.text)                                    # 마크다운 표

PPTX

In [ ]:
# [역할] PPT 표 도형 확인: 텍스트 상자는 조항 블록, 표 도형은 마크다운 표 블록으로 따로 나오는지 확인
from pptx import Presentation                        # PPT 만들기·읽기
from pptx.util import Inches                         # 도형 위치·크기 단위

prs = Presentation()                                 # 빈 프레젠테이션
slide = prs.slides.add_slide(prs.slide_layouts[6])   # 빈 레이아웃(6번) 슬라이드 추가
slide.shapes.add_textbox(Inches(0.5), Inches(0.5), Inches(6), Inches(1)).text_frame.text = "출장비 집행 현황"   # 제목 텍스트 상자
table = slide.shapes.add_table(3, 3, Inches(0.5), Inches(1.8), Inches(6), Inches(2)).table   # 3×3 표 도형
for r, row in enumerate([["부서", "1분기", "2분기"],          # 행마다
                         ["인프라사업부", "4200000", "3800000"],
                         ["구매팀", "1200000", "1400000"]]):
    for c, value in enumerate(row):                  # 칸마다
        table.cell(r, c).text = value                # 값 넣기
WITH_TABLE = SANDBOX / "with_table.pptx"             # 저장 경로
prs.save(str(WITH_TABLE))                            # 저장

doc = lp.parse_local(WITH_TABLE)                     # HANDLERS 를 거쳐 parse_pptx 로
print(f"블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count}  →  {summarize(chunk(doc))}")   # 블록·표 수 → 청크 요약
for b in doc.blocks:                                 # 블록마다
    print(f"--- [{b.kind}] {b.locator}")             # 종류와 위치
    print(b.text)                                    # 내용

In [ ]:
# [역할] 그룹 도형 확인: 그룹 안에 숨은 텍스트 상자·표도 _walk_shapes 로 펼쳐 읽히는지 확인
prs = Presentation()                                 # 빈 프레젠테이션
slide = prs.slides.add_slide(prs.slide_layouts[6])   # 빈 슬라이드
group = slide.shapes.add_group_shape()               # 그룹 도형 만들기
group.shapes.add_textbox(Inches(0.5), Inches(0.5), Inches(6), Inches(1)).text_frame.text = "숙박비 상한(그룹 안)"   # 그룹 안 텍스트 상자
frame = slide.shapes.add_table(2, 2, Inches(0.5), Inches(1.8), Inches(6), Inches(1))   # 2×2 표 (일단 슬라이드에)
for c, (head, value) in enumerate([("광역시", "100,000"), ("그 밖", "80,000")]):   # 열마다
    frame.table.cell(0, c).text = head               # 머리글
    frame.table.cell(1, c).text = value              # 값
group._element.append(frame._element)                # 표를 그룹 안으로 옮김 (python-pptx 에 공식 기능이 없어 XML 직접 이동)
GROUPED = SANDBOX / "grouped.pptx"                   # 저장 경로
prs.save(str(GROUPED))                               # 저장

print("맨 바깥 도형     :", [type(s).__name__ for s in Presentation(str(GROUPED)).slides[0].shapes])   # 겉에는 GroupShape 하나뿐
doc = lp.parse_pptx(GROUPED)                         # 파싱
print(f"블록 {len(doc.blocks)} · 표 {doc.table_count}")   # 그룹 안까지 읽었으면 블록 2 · 표 1
for b in doc.blocks:                                 # 블록마다
    print(f"--- [{b.kind}] {b.locator}")             # 종류와 위치
    print(b.text)                                    # 내용

In [ ]:
# [역할] 차트 확인: 차트 안의 데이터가 '항목 × 계열' 표 블록으로 꺼내지는지 확인
from pptx.chart.data import CategoryChartData       # 차트 데이터 만들기
from pptx.enum.chart import XL_CHART_TYPE           # 차트 종류

prs = Presentation()                                 # 빈 프레젠테이션
slide = prs.slides.add_slide(prs.slide_layouts[6])   # 빈 슬라이드
data = CategoryChartData()                           # 차트 데이터
data.categories = ["1분기", "2분기"]                   # 가로축 항목
data.add_series("출장비", (4200000, 3800000))          # 계열 1
data.add_series("교육비", (1200000, 1400000))          # 계열 2
chart = slide.shapes.add_chart(XL_CHART_TYPE.COLUMN_CLUSTERED, Inches(0.5), Inches(0.5),   # 묶은 세로 막대 차트
                               Inches(6), Inches(4), data).chart
chart.has_title = True                               # 차트 제목 켜기
chart.chart_title.text_frame.text = "분기별 집행액"     # 차트 제목
CHART = SANDBOX / "with_chart.pptx"                  # 저장 경로
prs.save(str(CHART))                                 # 저장

doc = lp.parse_pptx(CHART)                           # 파싱 (_pptx_chart 가 표로 변환)
print(f"블록 {len(doc.blocks)} · 표 {doc.table_count}  →  {summarize(chunk(doc))}")   # 블록·표 수 → 청크 요약
for b in doc.blocks:                                 # 블록마다
    print(f"--- [{b.kind}] {b.locator}")             # 종류와 위치 (슬라이드 1 · 차트1)
    print(b.text)                                    # 제목 + 데이터 표

PDF

In [ ]:
# [역할] PDF 파서 두 가지 비교: 표를 못 잡는 parse_pdf vs 표를 떼는 parse_pdf_tables, 그리고 같은 규정 docx 결과와 대조
for name, fn in (("parse_pdf       ", lp.parse_pdf), ("parse_pdf_tables", lp.parse_pdf_tables)):   # 두 파서를 차례로
    doc = fn(PDF)                                    # 같은 PDF 파싱
    print(f"{name}  블록 {len(doc.blocks):>2} · 쪽 {doc.page_count} · 표 {doc.table_count}  →  {summarize(chunk(doc))}")   # 결과 요약
docx = lp.parse_local(DOCX)                          # 같은 규정의 Word 원본 (정답 기준)
print(f"(같은 규정 .docx)  블록 {len(docx.blocks)} · 쪽 {docx.page_count} · 표 {docx.table_count}  →  {summarize(chunk(docx))}")   # 청크 수가 parse_pdf_tables 와 같으면 성공

In [ ]:
# [역할] 표 글자 중복 확인: 표 안의 숫자(100,000)가 표 블록에만 있고 본문 블록에는 중복되지 않는지 확인
doc = lp.parse_pdf_tables(PDF)                       # 표 분리 파싱
table = next(b for b in doc.blocks if b.kind == "표" and b.text.startswith("| 직급 | 일비"))   # 직급별 여비 표 찾기
body = "\n".join(b.text for b in doc.blocks if b.kind == "조항")   # 본문 블록만 모아 한 덩어리로

print(table.locator)                                 # 표 위치 (예: p.11 · 표1)
print(table.text)                                    # 표 내용
print()                                              # 빈 줄
print("「100,000」 이 표에    :", "100,000" in table.text)   # True 가 정상
print("「100,000」 이 본문에도 :", "100,000" in body)        # False 가 정상 (outside 필터로 표 글자 제외)

PDF 스캔본

In [ ]:
# [역할] 스캔본 PDF 확인: 쪽마다 그림 한 장뿐이라 로컬 라이브러리(pypdf·pdfplumber)로는 글자 0 → parse_local 이 막히는 것을 확인
import pdfplumber                                    # 표 인식 PDF 라이브러리
from pypdf import PdfReader                          # 기본 PDF 라이브러리
from app.core.exceptions import ValidationFailed     # 글자 없음 등 입력 검증 실패 예외

SCAN = SANDBOX / "DOC-HR-014_국내출장_여비_규정_v2.0_스캔본.pdf"   # 같은 규정을 스캔한 PDF
reader = PdfReader(str(SCAN))                        # 열기
print("쪽 수             :", len(reader.pages))       # 쪽 수
print("쪽마다 든 그림     :", [len(p.images) for p in reader.pages])   # 쪽마다 그림 1장 = 스캔본
print("pypdf 글자        :", sum(len((p.extract_text() or "").strip()) for p in reader.pages))   # 글자 수 → 0
with pdfplumber.open(str(SCAN)) as pdf:              # pdfplumber 로도 열어 보기
    print("pdfplumber 글자   :", sum(len((p.extract_text() or "").strip()) for p in pdf.pages),   # 글자 수 → 0
          "· 표", sum(len(p.find_tables()) for p in pdf.pages))                                   # 표 수 → 0
try:                                                 # 막히는 것을 보여 주기 위해
    lp.parse_local(SCAN)                             # 로컬 파싱 시도
except ValidationFailed as e:                        # 글자가 하나도 없으면
    print("parse_local       막힘 —", e)              # 오류 메시지 출력

상용 API OCR

In [ ]:
# [역할] 상용 OCR 확인: 업스테이지로 스캔본을 OCR(force) 해서 별표1 정답 숫자 10개가 몇 개 나오는지 확인. 키가 없으면 호출 안 함(비용 0)
from app.core.config import get_settings            # 설정 읽기

settings = get_settings()                            # 설정 객체 (.env 반영)
HAS_KEY = bool(settings.upstage_api_key)             # API 키가 있는지

TRUTH = ["40,000", "100,000", "80,000", "30,000", "35,000", "70,000", "27,000", "60,000", "25,000", "50,000"]   # 별표1 표의 정답 숫자들

if HAS_KEY:                                          # 키가 있을 때만 (호출하면 과금)
    from app.integrations.upstage import UpstageParser   # 상용 파서 어댑터

    before = settings.upstage_parse_ocr              # 원래 OCR 설정 기억
    settings.upstage_parse_ocr = "force"             # 모든 쪽을 OCR 하도록 잠시 변경
    try:
        ocr_doc = UpstageParser().parse(str(SCAN))   # 스캔본을 업스테이지로 파싱
    finally:                                         # 성공하든 실패하든
        settings.upstage_parse_ocr = before          # 설정을 원래대로 복구
    text = "\n".join(b.text for b in ocr_doc.blocks)   # 모든 블록 글자 합치기
    print(f"OCR  블록 {len(ocr_doc.blocks)} · 쪽 {ocr_doc.page_count} · 표 {ocr_doc.table_count} · 글자 {len(text)}")   # 결과 요약
    print("별표1 값 중 OCR 에 나온 것 :", sum(v in text for v in TRUTH), "/", len(TRUTH))   # 정답 숫자 몇 개를 읽었나
    print("설정은 원래대로 : ocr =", settings.upstage_parse_ocr)   # 복구 확인 (auto)
else:                                                # 키가 없으면
    print("키가 없습니다 — 이 셀은 부르지 않고 지나갑니다(비용 0)")   # 건너뜀 안내